# 9. How deep should the tree grow?

**Open:** restart R; save the supplied script as `scripts/my-09-choose-tree.R`.

**Input:** Chapter 8's `tree-input.rds`. **Result:** one selected tree, before its test score is calculated.

## 9.1 Make a prediction about complexity

The invented example fitted its training records perfectly. Would allowing more splits make predictions for unseen farms better too?

**Write your guess:** a deeper tree's training error will usually decrease, but what might happen on farms it did not fit?

Run the supplied `load-tree-input` block:

In [1]:
source("scripts/tree_helpers.R")
require_tree_package()
if (!file.exists("outputs/checkpoints/tree-input.rds")) stop("Run Chapter 8 first to create tree-input.rds.")
tree_input <- readRDS("outputs/checkpoints/tree-input.rds")
train_survey <- tree_input$train

## 9.2 Tell R what to predict and what it may use

Run `tree-formula`:

In [2]:
tree_formula <- log10_msc ~ pasture_hours + milking_cows
tree_formula
candidate_depths <- 1:3

log10_msc ~ pasture_hours + milking_cows

Read `~` as “predict using”. Here `+` lists two candidate inputs; it does not add the hours to the cow count. `1:3` lists 1, 2 and 3. The tree may use either variable, both, or neither if no split is useful.

We compare maximum depths **1, 2 and 3**. Depth counts splits along one root-to-leaf path, not the total number of splits in the whole tree. Depth 1 allows one split from the root, giving at most two leaves. Depth 2 can split those groups again. The software learns cut positions; we choose the allowed depth using validation.

## 9.3 Hide training farms in turn

We already have **81 training farms** and **21 test farms**. To choose depth, divide only the 81 training farms into five batches, called **folds**. Fit using four batches and check predictions on the fifth; rotate until every training record has been checked once.

```mermaid
flowchart LR
    accTitle: Choose Complexity Inside Training Farms
    accDescr: Only training farms rotate through fitting and validation to choose depth. The final test farms are reserved for Chapter 10.
    training_farms["81 training farms"] --> rotate_folds["Rotate five validation batches"]
    rotate_folds --> choose_depth["Choose depth using errors"]
    choose_depth --> final_fit["Refit on all 81 farms"]
```

**Check:** all rounds from a farm stay in the same batch. A hidden batch contains temporarily held-out **training** farms. The 21 final test farms take no part in choosing depth.

Run the supplied `training-folds` block:

In [3]:
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(2027)
training_farms <- sort(unique(train_survey$FarmID), method = "radix")
fold_labels <- sample(rep(1:5, length.out = length(training_farms)))
fold_table <- data.frame(FarmID = training_farms, fold = fold_labels)
row_fold <- fold_table$fold[match(train_survey$FarmID, fold_table$FarmID)]
stopifnot(!any(fold_table$FarmID %in% tree_input$test$FarmID))
data.frame(fold = 1:5, validation_farms = as.integer(table(fold_labels)),
           validation_records = as.integer(table(row_fold)))

fold,validation_farms,validation_records
<int>,<int>,<int>
1,17,84
2,16,86
3,16,86
4,16,83
5,16,81


The counts differ slightly across batches. The final score pools all 420 out-of-fold errors, giving each record equal weight; it is not an unweighted average of five RMSEs.

## 9.4 Choose using predictions for hidden farms

Run `validate-depth`. Its supplied loop learns medians inside each fitting batch, fills its hidden batch using those same medians, fits each candidate and stores the predictions. You may expand the implementation; you need to explain the information flow.

In [4]:
# Each record is predicted once by a fit that has never seen its farm.
oof <- matrix(NA_real_, nrow(train_survey), length(candidate_depths))
oof_baseline <- rep(NA_real_, nrow(train_survey))
fold_medians <- list()
validation_nodes <- list()
for (fold in 1:5) {
  fitting <- train_survey[row_fold != fold, ]
  validation <- train_survey[row_fold == fold, ]
  stopifnot(length(intersect(fitting$FarmID, validation$FarmID)) == 0)
  medians <- learn_medians(fitting)
  fold_medians[[fold]] <- data.frame(fold = fold, feature = tree_features, median = unname(medians))
  fitting <- fill_predictors(fitting, medians)
  validation <- fill_predictors(validation, medians)
  oof_baseline[row_fold == fold] <- mean(fitting$log10_msc)
  for (j in seq_along(candidate_depths)) {
    candidate <- rpart::rpart(tree_formula, data = fitting, method = "anova",
                              control = tree_control(candidate_depths[j]))
    candidate_nodes <- extract_tree_nodes(candidate)
    candidate_nodes$fold <- fold
    candidate_nodes$depth <- candidate_depths[j]
    validation_nodes[[length(validation_nodes) + 1L]] <- candidate_nodes
    oof[row_fold == fold, j] <- predict(candidate, newdata = validation)
  }
}
stopifnot(all(is.finite(oof)), all(is.finite(oof_baseline)))
observed <- train_survey$log10_msc
validation_scores <- data.frame(depth = candidate_depths,
  OOF_RMSE_log10 = vapply(seq_along(candidate_depths), function(j) sqrt(mean((observed - oof[, j])^2)), numeric(1)),
  OOF_MAE_log10 = vapply(seq_along(candidate_depths), function(j) mean(abs(observed - oof[, j])), numeric(1)))
baseline_validation <- data.frame(model = "Training-fold mean", depth = 0,
  OOF_RMSE_log10 = sqrt(mean((observed - oof_baseline)^2)), OOF_MAE_log10 = mean(abs(observed - oof_baseline)))

Then run the supplied `choose-depth` block:

In [5]:
chosen_depth <- min(validation_scores$depth[validation_scores$OOF_RMSE_log10 <= min(validation_scores$OOF_RMSE_log10) + 1e-12])
validation_scores$selected <- validation_scores$depth == chosen_depth
validation_scores
baseline_validation
chosen_depth

depth,OOF_RMSE_log10,OOF_MAE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7219814,0.5646934,TRUE
2,0.7268655,0.5645752,FALSE
3,0.7423389,0.5842456,FALSE


model,depth,OOF_RMSE_log10,OOF_MAE_log10
<chr>,<dbl>,<dbl>,<dbl>
Training-fold mean,0,0.7206158,0.5574663


[1] 1

**OOF** means **out of fold**: each prediction came from a tree that had not seen that record's farm. We fixed **smallest OOF RMSE** as the selection rule, with the shallower tree winning a tie. MAE is also reported, but does not choose depth.

**Find:** which row is selected? Does it also beat the training-fold-mean baseline shown below the table? Selecting the best of three trees does not establish that it beats a simpler reference.

Do not compare these validation scores with Chapter 7's test RMSE: they concern different farms. Compare methods on the **same held-out records**. Validation and final test scores need not match.

## 9.5 Fit the selected rule and challenge your guess

Run `fit-tree`. `rpart::rpart()` calls the fitting function from the rpart package; `method = "anova"` specifies a numeric regression target. `tree_control()` supplies the fixed settings and selected depth. The fitted medians now use all training records:

In [6]:
training_medians <- learn_medians(train_survey)
train_filled <- fill_predictors(train_survey, training_medians)
final_tree <- rpart::rpart(tree_formula, data = train_filled, method = "anova",
                          control = tree_control(chosen_depth))
data.frame(feature = tree_features, training_median = unname(training_medians))

feature,training_median
<chr>,<dbl>
pasture_hours,9
milking_cows,72


Check the median table: these values came from the training records. `final_tree` is the R object storing the learned rules and leaf means; Chapter 10 draws and checks them.

Run the supplied `training-check` to compare fitted training errors with hidden-farm errors:

In [7]:
# An illustration after selection: these fits do not alter the chosen depth.
validation_scores$training_RMSE_log10 <- vapply(candidate_depths, function(depth) {
  fitted <- rpart::rpart(tree_formula, data = train_filled, method = "anova", control = tree_control(depth))
  sqrt(mean((train_filled$log10_msc - predict(fitted, newdata = train_filled))^2))
}, numeric(1))
validation_scores[c("depth", "training_RMSE_log10", "OOF_RMSE_log10", "selected")]

depth,training_RMSE_log10,OOF_RMSE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7014369,0.7219814,TRUE
2,0.6878793,0.7268655,FALSE
3,0.6767658,0.7423389,FALSE


**Return to your guess:** does a lower training error also give a lower OOF error here? Explain why choosing the smallest training error would answer the wrong question.

The extra training fits illustrate the difference; they do not change the depth already selected from OOF RMSE. We have not used a tree test score to choose predictors, depth or the missing-value rule.

## 9.6 Save the choice

Run `save-tree-fit`:

In [8]:
write.csv(fold_table, "outputs/tree/training-farm-folds.csv", row.names = FALSE)
write.csv(do.call(rbind, fold_medians), "outputs/tree/fold-medians.csv", row.names = FALSE)
write.csv(do.call(rbind, validation_nodes), "outputs/tree/validation-nodes.csv", row.names = FALSE)
write.csv(validation_scores, "outputs/tree/validation-scores.csv", row.names = FALSE)
write.csv(baseline_validation, "outputs/tree/baseline-validation.csv", row.names = FALSE)
oof_table <- data.frame(FarmID = train_survey$FarmID, Sampling = train_survey$Sampling,
  fold = row_fold, observed_log10 = observed, baseline_prediction = oof_baseline,
  depth_1 = oof[, 1], depth_2 = oof[, 2], depth_3 = oof[, 3])
write.csv(oof_table, "outputs/tree/out-of-fold-predictions.csv", row.names = FALSE)
saveRDS(list(fit = final_tree, medians = training_medians, chosen_depth = chosen_depth,
             train = train_survey, test = tree_input$test, audit = tree_input$audit,
             validation = validation_scores, baseline_validation = baseline_validation),
        "outputs/checkpoints/tree-fit.rds")
cat("Saved tree-fit.rds; no tree test score has been calculated.\n")

Saved tree-fit.rds; no tree test score has been calculated.


**Find:** `outputs/checkpoints/tree-fit.rds`, `outputs/tree/validation-scores.csv` and `outputs/tree/training-farm-folds.csv`.

**Ask AI:** give it the actual validation table and ask which depth follows the predeclared RMSE rule. Check its answer against the selected row. A persuasive explanation cannot justify switching criteria after seeing the scores.

**Next:** [predict, check and explain](10-test-tree.ipynb).